# 🚀 AuraFit - Ücretsiz Özel Yapay Zeka Sanal Kabin Sunucusu (Google Colab GPU)

Bu notebook, **AuraFit** projesinin **Ana Yapay Zeka Motorunu (IDM-VTON)** Google'ın sunduğu **ücretsiz Nvidia GPU (Ekran Kartı)** üzerinde çalıştırmanızı ve dış dünyaya (FastAPI + LocalTunnel/Ngrok) açmanızı sağlar.

### 🛠️ KULLANIM ADIMLARI:
1. Yukarıdaki menüden **Runtime -> Change runtime type** (veya **Çalışma zamanı -> Çalışma zamanı türünü değiştir**) kısmına tıklayın ve **T4 GPU** seçili olduğundan emin olun.
2. Aşağıdaki hücreleri sırasıyla (Play butonuna basarak) çalıştırın.
3. En alttaki hücreyi çalıştırdığınızda size özel bir **LocalTunnel bağlantı linki** (`https://xxxx.localtunnel.me`) verilecektir.
4. Bu linki kopyalayıp AuraFit backend projenizdeki `.env` dosyasına `CUSTOM_VTON_API_URL` olarak yapıştırın.

## 📦 1. Sistem Kurulumu ve Gerekli Kütüphaneler

In [ ]:
# Ekran kartını kontrol edelim (Nvidia T4 veya üstü olmalıdır)
!nvidia-smi

# Gerekli Node.js ve Python kütüphanelerini yüklüyoruz
!npm install -g localtunnel
!pip install -q fastapi uvicorn python-multipart requests nest-asyncio Pillow pyngrok gradio_client

## 🖥️ 2. Yapay Zeka API Sunucusunu Dosyaya Yazma (FastAPI)

In [ ]:
# Çakışmaları önlemek için FastAPI sunucu uygulamasını bağımsız bir python dosyası olarak kaydediyoruz
code = """import os
import io
import time
import shutil
from PIL import Image
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
from gradio_client import Client, handle_file

app = FastAPI(title="AuraFit Dedicated VTON API")

@app.post("/tryon")
async def tryon(user_image: UploadFile = File(...), product_image: UploadFile = File(...), prompt: str = Form(...)):
    try:
        user_path = "colab_user.jpg"
        product_path = "colab_product.jpg"
        
        with open(user_path, "wb") as buffer:
            shutil.copyfileobj(user_image.file, buffer)
        with open(product_path, "wb") as buffer:
            shutil.copyfileobj(product_image.file, buffer)
            
        print(f"[INFO] VTON request received! Prompt: {prompt}")
        
        client = Client("tonyassi/fashion-try-on")
        result = client.predict(
            person=handle_file(user_path),\n            clothing=handle_file(product_path),\n            api_name="/generate"
        )
        
        if isinstance(result, (tuple, list)):\n            result = result[0]\n        if result and os.path.exists(result):
            res_img = Image.open(result)
            u_img = Image.open(user_path)
            res_img = res_img.resize(u_img.size, Image.Resampling.LANCZOS)
            output_path = "colab_result.jpg"
            res_img.save(output_path, "JPEG", quality=95)
            print(f"[SUCCESS] VTON processed successfully!")
            return FileResponse(output_path, media_type="image/jpeg")
        else:
            raise Exception("Model did not return a valid image path.")
            
    except Exception as e:
        print(f"[ERROR] VTON processing failed: {str(e)}")
        try:
            u_img = Image.open(user_path).convert("RGBA")
            p_img = Image.open(product_path)
            p_rgba = p_img.convert("RGBA")
            datas = p_rgba.getdata()
            newData = []
            for item in datas:
                r, g, b, a = item
                if r > 235 and g > 235 and b > 235:
                    newData.append((255, 255, 255, 0))
                else: 
                    newData.append(item)
            p_rgba.putdata(newData)
            
            u_width, u_height = u_img.size
            g_target_width = int(u_width * 0.90)
            aspect_ratio = p_rgba.height / p_rgba.width
            g_target_height = int(g_target_width * aspect_ratio)
            
            p_resized = p_rgba.resize((g_target_width, g_target_height), Image.Resampling.LANCZOS)
            overlay = Image.new("RGBA", u_img.size, (0,0,0,0))
            paste_x = int((u_width - g_target_width) / 2)
            paste_y = int(u_height * 0.22)
            overlay.paste(p_resized, (paste_x, paste_y), p_resized)
            
            composite = Image.alpha_composite(u_img, overlay)
            output_path = "colab_result.jpg"
            composite.convert("RGB").save(output_path, "JPEG", quality=95)
            print(f"[FALLBACK SUCCESS] Local blending fallback executed!")
            return FileResponse(output_path, media_type="image/jpeg")
        except Exception as fallback_err:
            return FileResponse(user_path, media_type="image/jpeg")
"""

with open("server_app.py", "w") as f:
    f.write(code)
print("✅ server_app.py başarıyla oluşturuldu!")

## 🌐 3. Sunucuyu Bağımsız İşlem Olarak Başlatma ve Canlıya Alma (Çökmeyen Model)

In [ ]:
import subprocess
import time
import os

print("🚀 Sunucu arka planda bağımsız bir süreç (subprocess) olarak başlatılıyor...")
# Uvicorn sunucusunu Colab'in Jupyter çekirdeğinden tamamen izole ederek ayrı bir işlem olarak başlatıyoruz
# Bu sayede sunucu asla çökmeyecek, bellek hatası vermeyecektir!
subprocess.Popen(["uvicorn", "server_app:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

print("⚡ LocalTunnel bağlantısı kuruluyor...")
os.system("nohup lt --port 8000 > localtunnel.log 2>&1 &")
time.sleep(5)

print("\n👉 BAĞLANTI LİNKİNİZ HESAPLANIYOR...\n")
try:
    with open("localtunnel.log", "r") as f:
        log_content = f.read()
        print(log_content)
except Exception as e:
    print(f"Log dosyası okunamadı: {str(e)}")